<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Logistic Regression</h1></center>

This notebook covers **Logistic Regression** for binary classification.

### Template:
1. Information About Dataset
2. Data Visualization
3. Train-Test Split
4. Model
5. Evaluation of Model

### Topics:
1. [What is Logistic Regression?](#1)
2. [Binary Classification — Breast Cancer](#2)
3. [Evaluation Metrics for Classification](#3)

In [ ]:
import seaborn as sns
headers = ['#000000']
print("Header Color:")
sns.palplot(sns.color_palette(headers))

In [ ]:
log_colors = ['#6a0572', '#ab83a1', '#f2d7ee', '#e91e8c', '#310d20']
print("Logistic Regression Colors:")
sns.palplot(sns.color_palette(log_colors))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, log_loss,
                              classification_report, confusion_matrix,
                              RocCurveDisplay, PrecisionRecallDisplay)
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
np.random.seed(42)
print("Libraries loaded.")

## Let's Start!

<a id = "1"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">What is Logistic Regression?</h1></center>

## Theory

Logistic Regression models the **probability** that an observation belongs to a class. It applies the **sigmoid function** to the linear combination of features:

$$P(y=1|X) = \sigma(z) = \frac{1}{1 + e^{-z}}, \quad z = \beta_0 + \beta_1 X_1 + \cdots + \beta_p X_p$$

The sigmoid squashes any real number into [0, 1], making it interpretable as a probability.

## Decision Boundary

A threshold (usually 0.5) converts the probability to a class label:

$$\hat{y} = \begin{cases} 1 & \text{if } P(y=1|X) \geq 0.5 \\ 0 & \text{otherwise} \end{cases}$$

## Loss Function

Logistic Regression minimises **Binary Cross-Entropy (Log-Loss)**:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\right]$$

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'], ['\\(','\\)']]}});</SCRIPT>

In [ ]:
# Visualise the sigmoid function
z = np.linspace(-8, 8, 300)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 4), dpi=100)
plt.plot(z, sigmoid, color="#6a0572", lw=2.5)
plt.axhline(0.5, color="#e91e8c", ls="--", lw=1.5, label="Threshold = 0.5")
plt.axvline(0,   color="gray",    ls=":",  lw=1.2)
plt.xlabel("z (linear combination)")
plt.ylabel("σ(z) — Probability")
plt.title("Sigmoid Function")
plt.legend()
plt.show()

<a id = "2"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Breast Cancer Classification</h1></center>

<center><h1 style = "background:#6a0572 ;color:white;border:0;font-weight:bold">Information About Dataset</h1></center>

**Breast Cancer Wisconsin** — 569 samples, 30 numeric features, binary target.

| Attribute | Description |
|---|---|
| 30 features | Mean, SE, and worst of: radius, texture, perimeter, area, smoothness, etc. |
| target | 0 = malignant, 1 = benign |

In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
df = data.frame
df["target"] = data.target
print(f"Shape: {df.shape}")
print(f"Target classes: {data.target_names}")
print(f"\nClass distribution:")
print(df["target"].value_counts().rename({0:"malignant", 1:"benign"}))
df.sample(5)

In [ ]:
df.describe().T

<center><h1 style = "background:#ab83a1 ;color:black;border:0;font-weight:bold">Data Visualization</h1></center>

In [ ]:
# Top feature distributions by class
top_features = ["mean radius", "mean texture", "mean area", "mean smoothness"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=100)

for ax, feat in zip(axes.ravel(), top_features):
    for label, color in [(0, "#6a0572"), (1, "#e91e8c")]:
        sns.kdeplot(df[df["target"]==label][feat], ax=ax,
                    label=data.target_names[label], color=color, fill=True, alpha=0.4)
    ax.set_title(feat)
    ax.legend()

plt.suptitle("Feature Distributions by Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (top features)
top10 = data.data.iloc[:, :10]
top10["target"] = data.target
plt.figure(figsize=(10, 7), dpi=100)
sns.heatmap(top10.corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="RdPu", linewidths=0.5)
plt.title("Correlation Heatmap (first 10 features)")
plt.show()

<center><h1 style = "background:#f2d7ee ;color:black;border:0;font-weight:bold">Train-Test Split</h1></center>

In [ ]:
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s  = sc.transform(X_test)

print(f"Total samples   : {len(X)}")
print(f"Training samples: {len(X_train)}  shape={X_train.shape}")
print(f"Test samples    : {len(X_test)}   shape={X_test.shape}")
print(f"Positive rate — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")

<center><h1 style = "background:#e91e8c ;color:white;border:0;font-weight:bold">Logistic Regression Model</h1></center>

**Key parameters:**
- `C` — inverse regularisation strength (smaller C = more regularisation)
- `solver` — optimisation algorithm (`lbfgs` default; `saga` for large datasets)
- `max_iter` — maximum iterations for convergence
- `class_weight` — use `"balanced"` for imbalanced datasets

In [ ]:
model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)
pred  = model.predict(X_test_s)
prob  = model.predict_proba(X_test_s)[:, 1]

train_score = model.score(X_train_s, y_train)
test_score  = model.score(X_test_s,  y_test)
print(f"Train Accuracy: {train_score:.4f}")
print(f"Test  Accuracy: {test_score:.4f}")

<a id = "3"></a>
<center><h1 style = "background:#310d20 ;color:white;border:0;font-weight:bold">Evaluation of Model</h1></center>

## Classification Metrics

| Metric | Formula | When to use |
|---|---|---|
| Accuracy | (TP+TN)/(TP+TN+FP+FN) | Balanced classes |
| Precision | TP/(TP+FP) | Minimise false positives |
| Recall | TP/(TP+FN) | Minimise false negatives (e.g., cancer) |
| F1 | 2·P·R/(P+R) | Balance precision and recall |
| ROC-AUC | Area under ROC curve | Overall discrimination ability |
| Log-Loss | Cross-entropy of probabilities | Calibration quality |

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'], ['\\(','\\)']]}});</SCRIPT>

In [ ]:
acc   = accuracy_score(y_test, pred)
prec  = precision_score(y_test, pred)
rec   = recall_score(y_test, pred)
f1    = f1_score(y_test, pred)
auc   = roc_auc_score(y_test, prob)
ll    = log_loss(y_test, prob)

results = [acc, prec, rec, f1, auc, ll]
metric_names = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Log-Loss"]
pd.DataFrame({"Metric": metric_names, "Score": results}).round(4)

In [ ]:
print(classification_report(y_test, pred, target_names=data.target_names))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=100)

# Confusion matrix
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="RdPu",
            xticklabels=data.target_names,
            yticklabels=data.target_names, ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

# ROC curve
RocCurveDisplay.from_predictions(y_test, prob, ax=axes[1], color="#6a0572")
axes[1].set_title("ROC Curve")

# Precision-Recall curve
PrecisionRecallDisplay.from_predictions(y_test, prob, ax=axes[2], color="#e91e8c")
axes[2].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.show()

In [ ]:
# Threshold tuning
thresholds = np.linspace(0.1, 0.9, 81)
f1s = [f1_score(y_test, (prob >= t).astype(int)) for t in thresholds]
best_t = thresholds[np.argmax(f1s)]

plt.figure(figsize=(9, 4), dpi=100)
plt.plot(thresholds, f1s, color="#6a0572", lw=2)
plt.axvline(best_t, color="#e91e8c", ls="--", label=f"Best threshold = {best_t:.2f}")
plt.axvline(0.5,    color="gray",    ls=":",  label="Default = 0.5")
plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.title("F1 Score vs Classification Threshold")
plt.legend()
plt.show()
print(f"Best threshold: {best_t:.2f}  →  F1 = {max(f1s):.4f}")

In [ ]:
# Cross-validation
from sklearn.pipeline import make_pipeline
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000, random_state=42))
cv_scores = cross_val_score(pipe, X, y, cv=skf, scoring="roc_auc")
print(f"5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Per-fold: {cv_scores.round(4)}")